In [ ]:
"""
=============================================================================
INTERPOL 2026 — PYTHON FLASK DASHBOARD
=============================================================================
HOW TO RUN:
    python interpol_dashboard.py

Then open in browser:
    http://127.0.0.1:5050

WHAT IT DOES:
    • Loads interpol_final_fugitive_db.csv (output of interpol_full_pipeline.py)
    • Trains all 3 pillars in memory on startup
    • Serves a full interactive web dashboard with:
        - KPI strip (6,479 fugitives, crime breakdown, model scores)
        - Live screening form (type a client name → get real-time risk score)
        - Pillar 2 cluster breakdown table
        - Recent screenings log
        - Matplotlib charts served as base64 PNGs

REQUIRES: Flask, matplotlib, pandas, numpy, scikit-learn (all available)
NO INTERNET NEEDED. NO STREAMLIT. NO PLOTLY.
=============================================================================
"""

import io, base64, os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from flask import Flask, render_template_string, request, jsonify

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier,
                               GradientBoostingClassifier,
                               IsolationForest)
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

# ─────────────────────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────────────────────
DB_PATH  = "/Users/dreyw/SMU_School/Term4/Project/Interpol/outputs/interpol_final_fugitive_db.csv"
RAW_PATH = "/Users/dreyw/SMU_School/Term4/Project/Interpol/crime_analysis_results_aft_transformer_ner.csv"

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
CRIME_WEIGHTS = {
    'terrorism':1.0,'homicide':0.9,'narcotics':0.7,'assault':0.6,
    'financial crime':0.5,'theft':0.4,'cyber crime':0.3,'Unknown':0.1,
}
CRIME_CLUSTER_LABELS = {
    0:'Assault',1:'Terrorist Organisation',
    2:'Homicide',3:'Transnational Terrorism',
    4:'Violent & Mixed Offences',
}
CRIME_CLUSTER_WEIGHTS = {0:0.6,1:1.0,2:0.9,3:1.0,4:0.7}
DARK_HAIR  = ['BLA','OTHD','BROF']
BROWN_HAIR = ['BRO','GRY','GRYG']
DARK_EYE   = ['BLA','BROD','OTHD','BROH','BROL']
BROWN_EYE  = ['BRO','GRY']
FEATURES   = ['age_delta_years','gender_match','height_delta_cm',
              'eye_color_match','hair_color_match','marks_match','sbert_name_score']
OPTIMAL_K  = 5

CRIME_HEX = {
    'terrorism':'#ff2d2d','homicide':'#ff6b2d','narcotics':'#f59e0b',
    'assault':'#eab308','financial crime':'#38bdf8',
    'theft':'#a78bfa','cyber crime':'#34d399','Unknown':'#64748b',
}
CLUSTER_HEX = ['#eab308','#ff2d2d','#ff6b2d','#ff2d2d','#f59e0b']

DARK_BG  = '#040810'
PANEL_BG = '#06090f'
GRID_CLR = '#1e2535'
TEXT_CLR = '#94a3b8'

# ─────────────────────────────────────────────────────────────────────────────
# STARTUP: LOAD DATA + TRAIN ALL PILLARS
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("  INTERPOL 2026 DASHBOARD — STARTUP")
print("=" * 60)

# Load final fugitive DB
db = pd.read_csv(DB_PATH)
db['sanctions_clean'] = db['sanctions_clean'].fillna('unknown offense')
db['name']            = db['name'].fillna('UNKNOWN')
db['crime_weight']    = db['detected_crime_type'].map(CRIME_WEIGHTS).fillna(0.1)
print(f"✅  Loaded {len(db):,} fugitives from {DB_PATH}")

# ── PILLAR 1: SBERT name embeddings ──────────────────────────────────────────
print("⚙  Building Pillar 1 name embeddings...")
name_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4),
                            max_features=5000, sublinear_tf=True)
name_mat  = name_vec.fit_transform(db['name'].str.upper().astype(str))
name_emb  = normalize(name_mat)
print(f"   Name embeddings: {name_emb.shape}")

def query_sbert(client_name: str) -> np.ndarray:
    """Return cosine similarity array against all fugitives."""
    q = normalize(name_vec.transform([client_name.upper()]))
    return cosine_similarity(q, name_emb)[0]

# ── PILLAR 2: K-Means on sanctions text ──────────────────────────────────────
print("⚙  Building Pillar 2 crime cluster embeddings...")
sanction_pipe = make_pipeline(
    TfidfVectorizer(max_features=1000, ngram_range=(1,2),
                    stop_words='english', min_df=2, sublinear_tf=True),
    TruncatedSVD(n_components=50, random_state=42),
)
X_sanction = sanction_pipe.fit_transform(db['sanctions_clean'].astype(str))
kmeans     = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
db['K_Cluster']           = kmeans.fit_predict(X_sanction)
db['K_Cluster_label']     = db['K_Cluster'].map(CRIME_CLUSTER_LABELS)
db['cluster_risk_weight'] = db['K_Cluster'].map(CRIME_CLUSTER_WEIGHTS)
km_sil = silhouette_score(X_sanction, db['K_Cluster'])
print(f"   K-Means silhouette: {km_sil:.3f}")

# ── PILLAR 3: Biometric Refutation — generate pairs + train ──────────────────
print("⚙  Training Pillar 3 biometric refutation models...")
np.random.seed(42)

fug_pool = db.dropna(subset=['GENDER','age_today']).reset_index(drop=True)
records  = []
for _ in range(2000):
    fug      = fug_pool.sample(1).iloc[0]
    is_match = np.random.choice([1,0], p=[0.35,0.65])
    if is_match:
        rec = dict(age_delta=abs(np.random.normal(0.5,1.0)),
                   gender_match=1,
                   height_delta=abs(np.random.normal(0,1.5)),
                   eye=np.random.choice([1,0],p=[0.92,0.08]),
                   hair=np.random.choice([1,0],p=[0.87,0.13]),
                   marks=np.random.choice([1,0],p=[0.82,0.18]),
                   sbert=np.random.uniform(0.83,0.99), label=1)
    else:
        gm = np.random.choice([1,0],p=[0.20,0.80])
        rec = dict(age_delta=np.random.choice([1,3,8,15,25],p=[0.05,0.15,0.30,0.30,0.20]),
                   gender_match=gm,
                   height_delta=abs(np.random.normal(9,7)),
                   eye=np.random.choice([1,0],p=[0.25,0.75]),
                   hair=np.random.choice([1,0],p=[0.28,0.72]),
                   marks=np.random.choice([1,0],p=[0.18,0.82]),
                   sbert=np.random.uniform(0.68,0.91), label=0 if not gm else 0)
    records.append(rec)

train_df = pd.DataFrame(records)
train_df.columns = FEATURES + ['label']
X_tr, X_te, y_tr, y_te = train_test_split(
    train_df[FEATURES], train_df['label'], test_size=0.25, random_state=42, stratify=train_df['label'])

scaler    = StandardScaler()
X_tr_sc   = scaler.fit_transform(X_tr)
X_te_sc   = scaler.transform(X_te)

models = {
    'LR':  LogisticRegression(random_state=42, max_iter=1000),
    'DT':  DecisionTreeClassifier(max_depth=5, random_state=42),
    'RF':  RandomForestClassifier(n_estimators=150, random_state=42),
    'GB':  GradientBoostingClassifier(n_estimators=150, learning_rate=0.08,
                                       max_depth=4, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
}
p3 = {}
for name, clf in models.items():
    Xt = X_tr_sc if name in ['LR','SVM'] else X_tr
    clf.fit(Xt, y_tr)
    Xv = X_te_sc if name in ['LR','SVM'] else X_te
    yp = clf.predict(Xv)
    yprob = clf.predict_proba(Xv)[:,1]
    from sklearn.metrics import roc_auc_score, f1_score
    p3[name] = dict(model=clf,
                    auc=roc_auc_score(y_te,yprob),
                    f1=f1_score(y_te,yp))

iso = IsolationForest(contamination=0.15, random_state=42, n_estimators=150)
iso.fit(X_tr[y_tr==1])
gb_model = p3['GB']['model']

# Feature importance
feat_imp = dict(zip(FEATURES, gb_model.feature_importances_))

print(f"   Models trained: {list(p3.keys())}")
print(f"   GB AUC: {p3['GB']['auc']:.4f} | RF AUC: {p3['RF']['auc']:.4f}")

# ── In-memory screening log ───────────────────────────────────────────────────
screening_log = []

print("\n✅  All pillars ready. Starting Flask server...")
print("   Open browser at: http://127.0.0.1:5050\n")

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: SCREEN A CLIENT
# ─────────────────────────────────────────────────────────────────────────────

def screen_client(client_name, client_age, client_gender, client_height,
                  client_eye_enc, client_hair_enc, marks_match,
                  linkage_score=0.1, visual_score=0.7):
    """Full pipeline: Pillar 1 → Pillar 3 → final criticality score."""
    # Pillar 1
    sims    = query_sbert(client_name)
    top3_idx = np.argsort(sims)[::-1][:3]
    top_idx  = int(top3_idx[0])
    sbert_sc = float(sims[top_idx])
    fug      = db.iloc[top_idx]

    # Pillar 3 features
    age_delta    = abs(client_age - fug['age_today'])
    gender_match = 1 if client_gender == fug['GENDER'] else 0
    fug_ht       = fug['height_cm']
    height_delta = abs(client_height - fug_ht) if pd.notna(fug_ht) else 3.0

    feat = pd.DataFrame([[age_delta, gender_match, height_delta,
                          client_eye_enc, client_hair_enc, marks_match,
                          sbert_sc]], columns=FEATURES)
    feat_sc = scaler.transform(feat)

    gb_p  = float(gb_model.predict_proba(feat)[0][1])
    rf_p  = float(p3['RF']['model'].predict_proba(feat)[0][1])
    lr_p  = float(p3['LR']['model'].predict_proba(feat_sc)[0][1])
    svm_p = float(p3['SVM']['model'].predict_proba(feat_sc)[0][1])

    # Hard gate
    if gender_match == 0:
        ref_score = 0.0
        verdict   = 'AUTO-CLEARED'
        reason    = 'Gender mismatch — hard gate triggered'
    elif gb_p >= 0.70:
        ref_score = gb_p
        verdict   = 'HIGH RISK'
        reason    = 'Biometrics confirm identity match'
    elif gb_p >= 0.45:
        ref_score = gb_p
        verdict   = 'REVIEW'
        reason    = 'Borderline — manual analyst review required'
    elif age_delta > 10:
        ref_score = gb_p
        verdict   = 'REFUTED'
        reason    = f'Age gap {age_delta:.0f}yr exceeds 10yr threshold'
    else:
        ref_score = gb_p
        verdict   = 'REFUTED'
        reason    = 'Biometric anomaly score exceeds threshold'

    crime_w    = float(fug['cluster_risk_weight'])
    raw_score  = sbert_sc*0.40 + crime_w*0.30 + linkage_score*0.20 + visual_score*0.10
    final_risk = raw_score * ref_score

    # Top 3 matches
    top3 = []
    for i in top3_idx:
        row = db.iloc[int(i)]
        top3.append(dict(name=row['name'], score=round(float(sims[i]),4),
                         crime=row['detected_crime_type'], country=row.get('label','?')))

    return dict(
        client_name   = client_name,
        matched_name  = fug['name'],
        matched_id    = fug['id'],
        matched_crime = fug['detected_crime_type'],
        matched_country = fug.get('label','?'),
        matched_cluster = fug['K_Cluster_label'],
        sbert_score   = round(sbert_sc,4),
        crime_weight  = round(crime_w,2),
        refutation    = round(ref_score,4),
        final_risk    = round(final_risk,4),
        verdict       = verdict,
        reason        = reason,
        gb=round(gb_p,4), rf=round(rf_p,4),
        lr=round(lr_p,4), svm=round(svm_p,4),
        age_delta     = round(age_delta,1),
        gender_match  = gender_match,
        height_delta  = round(height_delta,1),
        top3          = top3,
    )

# ─────────────────────────────────────────────────────────────────────────────
# CHART GENERATORS (matplotlib → base64 PNG)
# ─────────────────────────────────────────────────────────────────────────────

def fig_to_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight', facecolor=DARK_BG)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return b64

def chart_crime_distribution():
    crime_counts = db['detected_crime_type'].value_counts()
    fig, ax = plt.subplots(figsize=(7,3.5), facecolor=DARK_BG)
    ax.set_facecolor(PANEL_BG)
    colors = [CRIME_HEX.get(c,'#64748b') for c in crime_counts.index]
    bars = ax.barh(range(len(crime_counts)), crime_counts.values,
                   color=colors, edgecolor=DARK_BG, linewidth=1)
    ax.set_yticks(range(len(crime_counts)))
    ax.set_yticklabels(crime_counts.index, color=TEXT_CLR, fontfamily='monospace', fontsize=9)
    ax.set_title('Crime Type Distribution', color='#e2e8f0',
                 fontfamily='monospace', fontsize=10, pad=8)
    ax.set_xlabel('Fugitives', color=TEXT_CLR, fontsize=9)
    ax.tick_params(colors=TEXT_CLR, labelsize=8)
    ax.grid(True, color=GRID_CLR, linewidth=0.5, axis='x', alpha=0.5)
    for spine in ax.spines.values(): spine.set_edgecolor(GRID_CLR)
    for bar, val in zip(bars, crime_counts.values):
        ax.text(val+10, bar.get_y()+bar.get_height()/2,
                f'{val:,}', va='center', color='#e2e8f0',
                fontsize=8, fontfamily='monospace')
    fig.tight_layout()
    return fig_to_b64(fig)

def chart_cluster_breakdown():
    fig, axes = plt.subplots(1, 2, figsize=(10,3.5), facecolor=DARK_BG)
    # Left: cluster sizes
    ax = axes[0]
    ax.set_facecolor(PANEL_BG)
    counts = db['K_Cluster'].value_counts().sort_index()
    bars = ax.bar(range(OPTIMAL_K), counts.values, color=CLUSTER_HEX,
                  edgecolor=DARK_BG, linewidth=1.5)
    ax.set_xticks(range(OPTIMAL_K))
    ax.set_xticklabels([f'C{i}\n{CRIME_CLUSTER_LABELS[i][:11]}' for i in range(OPTIMAL_K)],
                        color=TEXT_CLR, fontsize=7)
    ax.set_title('K-Means Crime Clusters (k=5)', color='#e2e8f0',
                 fontfamily='monospace', fontsize=10, pad=8)
    ax.set_ylabel('Fugitives', color=TEXT_CLR, fontsize=9)
    ax.tick_params(colors=TEXT_CLR, labelsize=8)
    ax.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.3, axis='y')
    for spine in ax.spines.values(): spine.set_edgecolor(GRID_CLR)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
                str(val), ha='center', va='bottom', color='#e2e8f0',
                fontsize=8, fontfamily='monospace')
    # Right: risk weight donut
    ax2 = axes[1]
    ax2.set_facecolor(PANEL_BG)
    rw_counts = db['cluster_risk_weight'].value_counts()
    wedge_colors = ['#ff2d2d','#ff6b2d','#f59e0b','#eab308']
    wedges, texts, autotexts = ax2.pie(
        rw_counts.values, labels=[f'w={v}' for v in rw_counts.index],
        colors=wedge_colors[:len(rw_counts)], autopct='%1.0f%%',
        pctdistance=0.75, startangle=90,
        wedgeprops=dict(width=0.5, edgecolor=DARK_BG, linewidth=2))
    for t in texts: t.set_color(TEXT_CLR); t.set_fontsize(8)
    for at in autotexts: at.set_color('white'); at.set_fontsize(8)
    ax2.set_title('Risk Weight Distribution', color='#e2e8f0',
                  fontfamily='monospace', fontsize=10, pad=8)
    for spine in ax2.spines.values(): spine.set_edgecolor(GRID_CLR)
    fig.tight_layout(pad=1.5)
    return fig_to_b64(fig)

def chart_model_performance():
    fig, axes = plt.subplots(1, 2, figsize=(10,3.5), facecolor=DARK_BG)
    names  = list(p3.keys())
    aucs   = [p3[m]['auc'] for m in names]
    f1s    = [p3[m]['f1']  for m in names]
    colors = ['#38bdf8','#a78bfa','#34d399','#fb923c','#f472b6']

    ax = axes[0]
    ax.set_facecolor(PANEL_BG)
    bars = ax.bar(range(len(names)), aucs, color=colors, edgecolor=DARK_BG, linewidth=1.5)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, color=TEXT_CLR, fontsize=9, fontfamily='monospace')
    ax.set_ylim(0.85, 1.02)
    ax.set_title('Pillar 3 — Model AUC-ROC', color='#e2e8f0',
                 fontfamily='monospace', fontsize=10, pad=8)
    ax.set_ylabel('AUC-ROC', color=TEXT_CLR, fontsize=9)
    ax.tick_params(colors=TEXT_CLR, labelsize=8)
    ax.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.3, axis='y')
    for spine in ax.spines.values(): spine.set_edgecolor(GRID_CLR)
    for bar, val in zip(bars, aucs):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.001, f'{val:.4f}',
                ha='center', va='bottom', color='#e2e8f0',
                fontsize=7.5, fontfamily='monospace')

    ax2 = axes[1]
    ax2.set_facecolor(PANEL_BG)
    imp_feat = [FEATURES[i] for i in np.argsort(gb_model.feature_importances_)]
    imp_vals = np.sort(gb_model.feature_importances_)
    bar_colors = ['#fb923c' if v > 0.5 else '#38bdf8' if v > 0.05 else '#475569'
                  for v in imp_vals]
    ax2.barh(range(len(imp_feat)), imp_vals, color=bar_colors, edgecolor=DARK_BG)
    ax2.set_yticks(range(len(imp_feat)))
    ax2.set_yticklabels(imp_feat, color=TEXT_CLR, fontsize=8, fontfamily='monospace')
    ax2.set_title('Pillar 3 — Feature Importance (GB)', color='#e2e8f0',
                  fontfamily='monospace', fontsize=10, pad=8)
    ax2.set_xlabel('Importance', color=TEXT_CLR, fontsize=9)
    ax2.tick_params(colors=TEXT_CLR, labelsize=8)
    ax2.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.3, axis='x')
    for spine in ax2.spines.values(): spine.set_edgecolor(GRID_CLR)

    fig.tight_layout(pad=1.5)
    return fig_to_b64(fig)

def chart_age_distribution():
    fig, ax = plt.subplots(figsize=(7,3.5), facecolor=DARK_BG)
    ax.set_facecolor(PANEL_BG)
    ages = db['age_today'].dropna()
    ax.hist(ages, bins=30, color='#38bdf8', edgecolor=DARK_BG,
            linewidth=0.8, alpha=0.85)
    ax.axvline(ages.mean(), color='#ff2d2d', linestyle='--',
               linewidth=1.5, label=f'Mean {ages.mean():.0f}yr')
    ax.set_title('Age Distribution of Fugitives', color='#e2e8f0',
                 fontfamily='monospace', fontsize=10, pad=8)
    ax.set_xlabel('Age (years)', color=TEXT_CLR, fontsize=9)
    ax.set_ylabel('Count', color=TEXT_CLR, fontsize=9)
    ax.tick_params(colors=TEXT_CLR, labelsize=8)
    ax.legend(fontsize=8, framealpha=0.2, labelcolor='white')
    ax.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.4)
    for spine in ax.spines.values(): spine.set_edgecolor(GRID_CLR)
    fig.tight_layout()
    return fig_to_b64(fig)

def chart_screening_result(result):
    """Mini chart for a single screening result."""
    fig, axes = plt.subplots(1, 3, figsize=(11,3), facecolor=DARK_BG)

    # 1: Model comparison bars
    ax = axes[0]
    ax.set_facecolor(PANEL_BG)
    mod_names = ['GB≈XGB','RF','LR','SVM']
    mod_vals  = [result['gb'],result['rf'],result['lr'],result['svm']]
    bar_clrs  = ['#ff2d2d' if v>=0.70 else '#f59e0b' if v>=0.45 else '#34d399'
                 for v in mod_vals]
    ax.barh(range(4), mod_vals, color=bar_clrs, edgecolor=DARK_BG)
    ax.set_yticks(range(4))
    ax.set_yticklabels(mod_names, color=TEXT_CLR, fontsize=9, fontfamily='monospace')
    ax.set_xlim(0,1.1)
    ax.axvline(0.70, color='#ff2d2d', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(0.45, color='#f59e0b', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title('Model Scores', color='#e2e8f0', fontsize=9, fontfamily='monospace')
    ax.tick_params(colors=TEXT_CLR, labelsize=8)
    ax.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.3, axis='x')
    for spine in ax.spines.values(): spine.set_edgecolor(GRID_CLR)
    for i,(val,) in enumerate(zip(mod_vals)):
        ax.text(val+0.01, i, f'{val:.3f}', va='center', color='#e2e8f0',
                fontsize=8, fontfamily='monospace')

    # 2: Pillar contribution breakdown
    ax2 = axes[1]
    ax2.set_facecolor(PANEL_BG)
    contrib = {
        'SBERT×40%': result['sbert_score']*0.40,
        'Crime×30%': result['crime_weight']*0.30,
        'Linkage×20%': 0.1*0.20,
        'Visual×10%': 0.7*0.10,
    }
    c_colors = ['#38bdf8','#ff2d2d','#a78bfa','#34d399']
    bars2 = ax2.bar(range(4), list(contrib.values()), color=c_colors,
                    edgecolor=DARK_BG, linewidth=1.5)
    ax2.set_xticks(range(4))
    ax2.set_xticklabels(list(contrib.keys()), color=TEXT_CLR,
                         fontsize=7, fontfamily='monospace', rotation=10)
    ax2.set_title('Pillar Contributions', color='#e2e8f0',
                  fontsize=9, fontfamily='monospace')
    ax2.set_ylabel('Score Component', color=TEXT_CLR, fontsize=8)
    ax2.tick_params(colors=TEXT_CLR, labelsize=8)
    ax2.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.3, axis='y')
    for spine in ax2.spines.values(): spine.set_edgecolor(GRID_CLR)
    for bar, val in zip(bars2, contrib.values()):
        ax2.text(bar.get_x()+bar.get_width()/2, val+0.003,
                 f'{val:.3f}', ha='center', va='bottom', color='#e2e8f0',
                 fontsize=8, fontfamily='monospace')

    # 3: Final risk gauge
    ax3 = axes[2]
    ax3.set_facecolor(PANEL_BG)
    risk  = result['final_risk']
    refut = result['refutation']
    raw   = risk/refut if refut > 0 else risk
    bars3 = ax3.barh(['Raw Score','Refutation','Final Risk'],
                     [min(raw,1), refut, risk],
                     color=['#64748b','#a78bfa','#ff2d2d' if risk>=0.70 else '#f59e0b' if risk>=0.45 else '#34d399'],
                     edgecolor=DARK_BG, linewidth=1.5)
    ax3.set_xlim(0,1.1)
    ax3.axvline(0.70, color='#ff2d2d', linestyle='--', linewidth=1, alpha=0.5)
    ax3.set_title('Risk Calculation', color='#e2e8f0',
                  fontsize=9, fontfamily='monospace')
    ax3.tick_params(colors=TEXT_CLR, labelsize=8)
    ax3.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.3, axis='x')
    for spine in ax3.spines.values(): spine.set_edgecolor(GRID_CLR)
    for bar, val in zip(bars3, [min(raw,1), refut, risk]):
        ax3.text(val+0.01, bar.get_y()+bar.get_height()/2,
                 f'{val:.4f}', va='center', color='#e2e8f0',
                 fontsize=9, fontfamily='monospace')

    fig.suptitle(f'Screening: {result["client_name"]} → {result["matched_name"]}',
                 color='#e2e8f0', fontfamily='monospace', fontsize=10, y=1.02)
    fig.tight_layout(pad=1.2)
    return fig_to_b64(fig)

# ─────────────────────────────────────────────────────────────────────────────
# PRE-COMPUTE STATIC CHARTS
# ─────────────────────────────────────────────────────────────────────────────
print("⚙  Pre-rendering charts...")
CHART_CRIME    = chart_crime_distribution()
CHART_CLUSTER  = chart_cluster_breakdown()
CHART_MODELS   = chart_model_performance()
CHART_AGE      = chart_age_distribution()
print("✅  Charts ready.\n")

# ─────────────────────────────────────────────────────────────────────────────
# KPI STATS
# ─────────────────────────────────────────────────────────────────────────────
KPI = dict(
    total       = f"{len(db):,}",
    terrorism   = f"{(db['detected_crime_type']=='terrorism').sum():,}",
    homicide    = f"{(db['detected_crime_type']=='homicide').sum():,}",
    countries   = f"{db['label'].nunique() if 'label' in db.columns else '—'}",
    km_sil      = f"{km_sil:.3f}",
    gb_auc      = f"{p3['GB']['auc']:.4f}",
)

# Cluster table data
CLUSTER_TABLE = []
for c in range(OPTIMAL_K):
    sub = db[db['K_Cluster']==c]
    top_crime = sub['detected_crime_type'].value_counts().index[0]
    w = float(CRIME_CLUSTER_WEIGHTS[c])
    CLUSTER_TABLE.append(dict(
        cluster   = int(c),
        label     = CRIME_CLUSTER_LABELS[c],
        count     = f"{len(sub):,}",
        weight    = w,
        weight_str= f"{w:.1f}",
        top_crime = top_crime,
        color     = CLUSTER_HEX[c],
        tier      = ('PROHIBITED' if w >= 1.0 else 'HIGH RISK' if w >= 0.7 else 'STANDARD'),
        tier_cls  = ('badge-red'  if w >= 1.0 else 'badge-amber' if w >= 0.7 else 'badge-blue'),
    ))

# ─────────────────────────────────────────────────────────────────────────────
# HTML TEMPLATE
# ─────────────────────────────────────────────────────────────────────────────

HTML = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>INTERPOL 2026 — Risk Intelligence Dashboard</title>
<link href="https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Rajdhani:wght@400;600;700&display=swap" rel="stylesheet">
<style>
  :root {
    --bg:      #040810;
    --panel:   #06090f;
    --border:  #1e2535;
    --text:    #94a3b8;
    --bright:  #e2e8f0;
    --red:     #ff2d2d;
    --amber:   #f59e0b;
    --blue:    #38bdf8;
    --green:   #00ff88;
    --purple:  #a78bfa;
    --mono:    'Share Tech Mono', monospace;
    --sans:    'Rajdhani', sans-serif;
  }
  * { box-sizing:border-box; margin:0; padding:0; }
  body {
    background: var(--bg);
    color: var(--text);
    font-family: var(--mono);
    min-height: 100vh;
    overflow-x: hidden;
  }

  /* scanlines overlay */
  body::before {
    content:'';
    position:fixed; inset:0; pointer-events:none; z-index:9999;
    background: repeating-linear-gradient(
      0deg, transparent, transparent 2px,
      rgba(0,0,0,0.04) 2px, rgba(0,0,0,0.04) 4px
    );
  }

  /* TOP NAV */
  nav {
    position: sticky; top:0; z-index:100;
    background: #040810ee;
    border-bottom: 1px solid var(--border);
    backdrop-filter: blur(8px);
    padding: 0 24px;
    display: flex; align-items: center; justify-content: space-between;
    height: 52px;
  }
  .nav-brand { display:flex; align-items:center; gap:14px; }
  .nav-logo {
    font-family: var(--sans); font-size:16px; font-weight:700;
    color: var(--bright); letter-spacing:2px;
  }
  .nav-logo span { color: var(--red); }
  .nav-sub { font-size:9px; color:#334155; letter-spacing:3px; }
  .nav-pills { display:flex; gap:6px; }
  .nav-pill {
    font-size:9px; letter-spacing:2px; text-transform:uppercase;
    padding:4px 14px; border-radius:3px; cursor:pointer;
    border:1px solid var(--border); color:#475569;
    background: transparent; font-family:var(--mono);
    transition: all 0.15s;
  }
  .nav-pill:hover, .nav-pill.active {
    background:#0d1f38; border-color:#3b82f640; color:var(--blue);
  }
  .nav-right { display:flex; align-items:center; gap:16px; }
  .live-dot {
    width:7px; height:7px; border-radius:50%;
    background:var(--green); box-shadow:0 0 8px var(--green);
    animation: pulse 2s infinite;
  }
  @keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.4} }
  .nav-time { font-size:10px; color:#334155; }
  .classified-badge {
    font-size:8px; letter-spacing:2px; padding:3px 10px; border-radius:3px;
    background:#ff2d2d18; border:1px solid #ff2d2d40; color:var(--red);
  }

  /* KPI STRIP */
  .kpi-strip {
    display:grid; grid-template-columns:repeat(6,1fr);
    border-bottom:1px solid var(--border); background:var(--panel);
  }
  .kpi-cell {
    padding:14px 18px; border-right:1px solid var(--border);
  }
  .kpi-cell:last-child { border-right:none; }
  .kpi-label { font-size:9px; color:#334155; letter-spacing:3px; margin-bottom:6px; }
  .kpi-value { font-size:22px; font-weight:700; font-family:var(--mono); }
  .kpi-sub { font-size:9px; color:#334155; margin-top:4px; }

  /* SECTIONS */
  .section { display:none; padding:20px 24px; }
  .section.active { display:block; }

  /* PAGE GRID */
  .grid-2 { display:grid; grid-template-columns:1fr 1fr; gap:16px; }
  .grid-3 { display:grid; grid-template-columns:1fr 1fr 1fr; gap:16px; }

  /* CARD */
  .card {
    background:var(--panel); border:1px solid var(--border);
    border-radius:6px; overflow:hidden;
  }
  .card-header {
    padding:12px 16px; border-bottom:1px solid var(--border);
    font-size:9px; color:#334155; letter-spacing:3px; text-transform:uppercase;
    display:flex; align-items:center; justify-content:space-between;
  }
  .card-body { padding:16px; }

  /* SCREENING FORM */
  .form-grid { display:grid; grid-template-columns:1fr 1fr; gap:12px; }
  .form-group { display:flex; flex-direction:column; gap:6px; }
  .form-label { font-size:10px; color:#475569; letter-spacing:2px; }
  .form-input, .form-select {
    background:#040810; border:1px solid var(--border);
    color:var(--bright); padding:8px 12px; border-radius:4px;
    font-family:var(--mono); font-size:12px;
    transition: border-color 0.15s;
  }
  .form-input:focus, .form-select:focus {
    outline:none; border-color:var(--blue);
  }
  .form-select option { background:#040810; }
  .btn-screen {
    width:100%; margin-top:14px; padding:12px;
    background:#0d1f38; border:1px solid #3b82f640;
    color:var(--blue); font-family:var(--mono); font-size:12px;
    letter-spacing:3px; cursor:pointer; border-radius:4px;
    transition:all 0.2s; text-transform:uppercase;
  }
  .btn-screen:hover { background:#1a3454; box-shadow:0 0 20px #38bdf820; }
  .btn-screen:disabled { opacity:0.4; cursor:not-allowed; }

  /* RESULT CARD */
  .result-verdict {
    padding:16px; border-radius:6px; margin-bottom:16px;
    border-left:4px solid;
  }
  .result-verdict.HIGH-RISK   { background:#ff2d2d12; border-color:#ff2d2d; }
  .result-verdict.REVIEW      { background:#f59e0b12; border-color:#f59e0b; }
  .result-verdict.REFUTED     { background:#00ff8812; border-color:#00ff88; }
  .result-verdict.AUTO-CLEARED{ background:#38bdf812; border-color:#38bdf8; }

  .verdict-tag {
    font-size:11px; font-weight:700; letter-spacing:3px; display:inline-block;
    padding:3px 10px; border-radius:3px; margin-bottom:10px;
  }
  .HIGH-RISK   .verdict-tag { background:#ff2d2d20; color:#ff2d2d; }
  .REVIEW      .verdict-tag { background:#f59e0b20; color:#f59e0b; }
  .REFUTED     .verdict-tag { background:#00ff8820; color:#00ff88; }
  .AUTO-CLEARED .verdict-tag { background:#38bdf820; color:#38bdf8; }

  .result-name  { font-size:18px; color:var(--bright); font-family:var(--sans); margin-bottom:4px; }
  .result-sub   { font-size:11px; color:#475569; }
  .result-score { font-size:32px; font-weight:700; font-family:var(--mono); }
  .result-reason{ font-size:11px; color:#64748b; margin-top:8px; }

  .metric-row {
    display:grid; grid-template-columns:repeat(4,1fr); gap:8px;
    margin-bottom:12px;
  }
  .metric-box {
    background:#040810; border:1px solid var(--border);
    border-radius:4px; padding:10px 12px;
  }
  .metric-label { font-size:9px; color:#334155; letter-spacing:2px; margin-bottom:4px; }
  .metric-val   { font-size:16px; font-family:var(--mono); }

  .top3-list { list-style:none; }
  .top3-item {
    display:flex; align-items:center; gap:10px;
    padding:8px 0; border-bottom:1px solid var(--border);
  }
  .top3-item:last-child { border-bottom:none; }
  .top3-rank { font-size:11px; color:#334155; min-width:24px; }
  .top3-name { font-size:12px; color:var(--bright); flex:1; }
  .top3-score{ font-size:12px; }
  .top3-crime{ font-size:10px; color:#475569; }

  /* CLUSTER TABLE */
  table { width:100%; border-collapse:collapse; }
  th {
    text-align:left; padding:10px 14px; font-size:9px;
    color:#334155; letter-spacing:3px; border-bottom:1px solid var(--border);
  }
  td { padding:10px 14px; font-size:12px; border-bottom:1px solid #0d1424; }
  tr:last-child td { border-bottom:none; }
  tr:hover td { background:#ffffff04; }
  .cluster-dot {
    width:8px; height:8px; border-radius:50%; display:inline-block;
    margin-right:8px; box-shadow:0 0 6px currentColor;
  }
  .risk-pill {
    font-size:10px; padding:2px 8px; border-radius:12px;
    font-family:var(--mono);
  }

  /* LOG TABLE */
  .log-row { display:flex; gap:0; border-bottom:1px solid #0d1424; }
  .log-row:hover { background:#ffffff03; }
  .log-cell { padding:9px 12px; font-size:11px; }
  .log-header { font-size:9px; color:#334155; letter-spacing:2px;
                border-bottom:1px solid var(--border); }

  /* VERDICT BADGES */
  .badge {
    font-size:9px; padding:2px 8px; border-radius:3px; letter-spacing:1px;
    font-family:var(--mono);
  }
  .badge-red    { background:#ff2d2d20; color:#ff2d2d; border:1px solid #ff2d2d30; }
  .badge-amber  { background:#f59e0b20; color:#f59e0b; border:1px solid #f59e0b30; }
  .badge-green  { background:#00ff8820; color:#00ff88; border:1px solid #00ff8830; }
  .badge-blue   { background:#38bdf820; color:#38bdf8; border:1px solid #38bdf830; }

  /* CHART IMG */
  .chart-img { width:100%; display:block; }

  /* LOADING */
  #loading {
    display:none; text-align:center; padding:30px;
    font-size:11px; color:#334155; letter-spacing:3px;
  }
  .spinner {
    display:inline-block; width:16px; height:16px;
    border:2px solid #1e2535; border-top-color:var(--blue);
    border-radius:50%; animation:spin 0.8s linear infinite;
    margin-right:8px; vertical-align:middle;
  }
  @keyframes spin { to{transform:rotate(360deg)} }

  /* MISC */
  .section-title {
    font-size:9px; color:#334155; letter-spacing:4px;
    text-transform:uppercase; margin-bottom:16px;
    display:flex; align-items:center; gap:10px;
  }
  .section-title::after {
    content:''; flex:1; height:1px; background:var(--border);
  }
  .glow-red    { color:var(--red);    text-shadow:0 0 20px #ff2d2d60; }
  .glow-amber  { color:var(--amber);  text-shadow:0 0 20px #f59e0b60; }
  .glow-blue   { color:var(--blue);   text-shadow:0 0 20px #38bdf860; }
  .glow-green  { color:var(--green);  text-shadow:0 0 20px #00ff8860; }
  .glow-purple { color:var(--purple); text-shadow:0 0 20px #a78bfa60; }

  .col-span-2 { grid-column: span 2; }
</style>
</head>
<body>

<!-- NAV -->
<nav>
  <div class="nav-brand">
    <div>
      <div class="nav-logo">INTERPOL <span>IRIS</span></div>
      <div class="nav-sub">AI-DRIVEN RISK INTELLIGENCE · 2026</div>
    </div>
    <div style="width:1px;height:30px;background:var(--border)"></div>
    <div class="nav-pills">
      <button class="nav-pill active" onclick="showTab('overview')">Overview</button>
      <button class="nav-pill" onclick="showTab('screening')">Live Screening</button>
      <button class="nav-pill" onclick="showTab('analytics')">Analytics</button>
      <button class="nav-pill" onclick="showTab('models')">Models</button>
      <button class="nav-pill" onclick="showTab('log')">Screening Log</button>
    </div>
  </div>
  <div class="nav-right">
    <div class="live-dot"></div>
    <span class="nav-time" id="clock">--:--:-- UTC</span>
    <div class="classified-badge">⬛ CLASSIFIED</div>
  </div>
</nav>

<!-- KPI STRIP -->
<div class="kpi-strip">
  <div class="kpi-cell">
    <div class="kpi-label">TOTAL FUGITIVES</div>
    <div class="kpi-value glow-red">{{ kpi.total }}</div>
    <div class="kpi-sub">Unique INTERPOL records</div>
  </div>
  <div class="kpi-cell">
    <div class="kpi-label">TERRORISM</div>
    <div class="kpi-value glow-red">{{ kpi.terrorism }}</div>
    <div class="kpi-sub">Weight 1.0 — Prohibited</div>
  </div>
  <div class="kpi-cell">
    <div class="kpi-label">HOMICIDE</div>
    <div class="kpi-value glow-amber">{{ kpi.homicide }}</div>
    <div class="kpi-sub">Weight 0.9 — Prohibited</div>
  </div>
  <div class="kpi-cell">
    <div class="kpi-label">JURISDICTIONS</div>
    <div class="kpi-value glow-blue">{{ kpi.countries }}</div>
    <div class="kpi-sub">Countries of operation</div>
  </div>
  <div class="kpi-cell">
    <div class="kpi-label">K-MEANS SILHOUETTE</div>
    <div class="kpi-value glow-green">{{ kpi.km_sil }}</div>
    <div class="kpi-sub">Pillar 2 cluster quality</div>
  </div>
  <div class="kpi-cell">
    <div class="kpi-label">GB AUC-ROC</div>
    <div class="kpi-value glow-purple">{{ kpi.gb_auc }}</div>
    <div class="kpi-sub">Pillar 3 best model</div>
  </div>
</div>

<!-- ═══════ OVERVIEW TAB ═══════ -->
<div id="tab-overview" class="section active">
  <div class="section-title">DATABASE OVERVIEW</div>
  <div class="grid-2" style="margin-bottom:16px">
    <div class="card">
      <div class="card-header">CRIME TYPE DISTRIBUTION — 6,479 FUGITIVES</div>
      <div class="card-body" style="padding:8px">
        <img class="chart-img" src="data:image/png;base64,{{ chart_crime }}">
      </div>
    </div>
    <div class="card">
      <div class="card-header">AGE DISTRIBUTION</div>
      <div class="card-body" style="padding:8px">
        <img class="chart-img" src="data:image/png;base64,{{ chart_age }}">
      </div>
    </div>
  </div>
  <div class="card">
    <div class="card-header">PILLAR 2 — K-MEANS CRIME CLUSTER BREAKDOWN</div>
    <div class="card-body" style="padding:8px">
      <img class="chart-img" src="data:image/png;base64,{{ chart_cluster }}">
    </div>
  </div>
  <br>
  <div class="card">
    <div class="card-header">CLUSTER REFERENCE TABLE</div>
    <table>
      <thead>
        <tr>
          <th>CLUSTER</th><th>LABEL</th><th>FUGITIVES</th>
          <th>RISK WEIGHT</th><th>TOP CRIME TYPE</th><th>TIER</th>
        </tr>
      </thead>
      <tbody>
        {% for row in cluster_table %}
        <tr>
          <td>
            <span class="cluster-dot" style="background:{{row.color}};color:{{row.color}}"></span>
            C{{ row.cluster }}
          </td>
          <td style="color:var(--bright)">{{ row.label }}</td>
          <td style="font-family:var(--mono)">{{ row.count }}</td>
          <td>
            <span class="risk-pill" style="background:{{row.color}}20;color:{{row.color}};border:1px solid {{row.color}}40">
              {{ row.weight_str }}
            </span>
          </td>
          <td style="color:#64748b">{{ row.top_crime }}</td>
          <td>
            <span class="badge {{ row.tier_cls }}">{{ row.tier }}</span>
          </td>
        </tr>
        {% endfor %}
      </tbody>
    </table>
  </div>
</div>

<!-- ═══════ SCREENING TAB ═══════ -->
<div id="tab-screening" class="section">
  <div class="section-title">LIVE CLIENT SCREENING</div>
  <div class="grid-2">

    <!-- LEFT: Form -->
    <div class="card">
      <div class="card-header">CLIENT SUBMISSION — ENTER DETAILS</div>
      <div class="card-body">
        <div class="form-grid">
          <div class="form-group col-span-2">
            <label class="form-label">CLIENT NAME (as submitted)</label>
            <input class="form-input" id="f-name" type="text"
                   placeholder="e.g. S. Nikitenko / Abdul Sambolotov" value="S. Nikitenko">
          </div>
          <div class="form-group">
            <label class="form-label">AGE</label>
            <input class="form-input" id="f-age" type="number" value="33" min="1" max="120">
          </div>
          <div class="form-group">
            <label class="form-label">GENDER</label>
            <select class="form-select" id="f-gender">
              <option value="M">Male (M)</option>
              <option value="F" selected>Female (F)</option>
            </select>
          </div>
          <div class="form-group">
            <label class="form-label">HEIGHT (cm)</label>
            <input class="form-input" id="f-height" type="number" value="165" min="100" max="220">
          </div>
          <div class="form-group">
            <label class="form-label">EYE COLOUR</label>
            <select class="form-select" id="f-eye">
              <option value="0">Dark (Black/Dark Brown)</option>
              <option value="1" selected>Brown/Grey</option>
              <option value="2">Light (Blue/Green)</option>
            </select>
          </div>
          <div class="form-group">
            <label class="form-label">HAIR COLOUR</label>
            <select class="form-select" id="f-hair">
              <option value="0">Dark (Black)</option>
              <option value="1" selected>Brown/Grey</option>
              <option value="2">Light (Blonde/Red)</option>
            </select>
          </div>
          <div class="form-group">
            <label class="form-label">DISTINGUISHING MARKS</label>
            <select class="form-select" id="f-marks">
              <option value="1">Match</option>
              <option value="0" selected>No Match / Unknown</option>
            </select>
          </div>
        </div>
        <button class="btn-screen" id="btn-run" onclick="runScreening()">
          ▶ RUN SCREENING
        </button>
        <div id="loading">
          <span class="spinner"></span>PROCESSING THROUGH PILLARS 1–3...
        </div>

        <!-- Quick demo buttons -->
        <div style="margin-top:16px;display:flex;gap:8px;flex-wrap:wrap">
          <div style="font-size:9px;color:#334155;letter-spacing:2px;width:100%;margin-bottom:4px">QUICK DEMOS:</div>
          <button class="nav-pill" onclick="setDemo('S. Nikitenko',33,'F',165,0,0,1)">Nikitenko</button>
          <button class="nav-pill" onclick="setDemo('Hasen Aksema',55,'M',175,1,1,0)">Aksema</button>
          <button class="nav-pill" onclick="setDemo('Abdul Sambolotov',34,'M',172,0,1,1)">Sambolotov</button>
          <button class="nav-pill" onclick="setDemo('M. Al-Saeed',40,'F',158,2,2,0)">Al-Saeed (Clear)</button>
          <button class="nav-pill" onclick="setDemo('JIAN XIA',48,'M',170,1,0,0)">Jian Xia</button>
          <button class="nav-pill" onclick="setDemo('Norbert Bialas',47,'M',180,1,1,0)">Bialas</button>
        </div>
      </div>
    </div>

    <!-- RIGHT: Result -->
    <div>
      <div id="result-area">
        <div class="card">
          <div class="card-body" style="text-align:center;padding:40px;color:#1e2535">
            <div style="font-size:40px;margin-bottom:12px">⬛</div>
            <div style="font-size:10px;letter-spacing:3px">AWAITING CLIENT SUBMISSION</div>
          </div>
        </div>
      </div>
    </div>
  </div>

  <!-- Chart area (shown after screening) -->
  <div id="screening-chart" style="display:none;margin-top:16px">
    <div class="card">
      <div class="card-header">SCREENING ANALYSIS CHARTS</div>
      <div class="card-body" style="padding:8px">
        <img class="chart-img" id="result-chart-img">
      </div>
    </div>
  </div>
</div>

<!-- ═══════ ANALYTICS TAB ═══════ -->
<div id="tab-analytics" class="section">
  <div class="section-title">PIPELINE ANALYTICS</div>
  <div class="card" style="margin-bottom:16px">
    <div class="card-header">PILLAR 2 — K-MEANS CRIME CLUSTERS + RISK WEIGHT DISTRIBUTION</div>
    <div class="card-body" style="padding:8px">
      <img class="chart-img" src="data:image/png;base64,{{ chart_cluster }}">
    </div>
  </div>
  <div class="grid-2">
    <div class="card">
      <div class="card-header">CRIME DISTRIBUTION</div>
      <div class="card-body" style="padding:8px">
        <img class="chart-img" src="data:image/png;base64,{{ chart_crime }}">
      </div>
    </div>
    <div class="card">
      <div class="card-header">AGE DISTRIBUTION</div>
      <div class="card-body" style="padding:8px">
        <img class="chart-img" src="data:image/png;base64,{{ chart_age }}">
      </div>
    </div>
  </div>
  <br>
  <!-- Pipeline formula card -->
  <div class="card">
    <div class="card-header">FINAL RISK FORMULA</div>
    <div class="card-body">
      <div style="font-size:14px;color:var(--bright);font-family:var(--mono);
                  text-align:center;padding:16px 0;line-height:2.2">
        <span style="color:var(--blue)">SBERT × 0.40</span>
        &nbsp;+&nbsp;
        <span style="color:var(--red)">Crime × 0.30</span>
        &nbsp;+&nbsp;
        <span style="color:var(--purple)">Linkage × 0.20</span>
        &nbsp;+&nbsp;
        <span style="color:var(--green)">Visual × 0.10</span>
        <br>
        <span style="color:#475569">× &nbsp;</span>
        <span style="color:var(--amber)">Refutation Score &nbsp;[0.0 → 1.0]</span>
        <br>
        <span style="font-size:11px;color:#334155">
          Refutation = 0.0 → Final Risk = 0.0 regardless of other pillars
        </span>
      </div>
      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-top:8px">
        {% for pillar in [
          {'name':'Pillar 1','label':'SBERT Identity','weight':'40%','color':'#38bdf8','desc':'Cosine similarity on name embeddings'},
          {'name':'Pillar 2','label':'Crime Severity','weight':'30%','color':'#ff2d2d','desc':'K-Means on sanctions text'},
          {'name':'Pillar 3','label':'Biometric Gate','weight':'Penalty','color':'#f59e0b','desc':'5 classifiers + hard gender gate'},
          {'name':'Pillars 4-5','label':'Linkage + Visual','weight':'30%','color':'#a78bfa','desc':'GraphSAGE + CLIP (future)'},
        ] %}
        <div style="background:#040810;border:1px solid var(--border);border-left:3px solid {{pillar.color}};
                    padding:12px;border-radius:4px">
          <div style="font-size:9px;color:#334155;letter-spacing:2px;margin-bottom:6px">{{pillar.name}}</div>
          <div style="font-size:13px;color:{{pillar.color}};font-family:var(--sans);font-weight:700">{{pillar.label}}</div>
          <div style="font-size:18px;color:{{pillar.color}};font-family:var(--mono);margin:4px 0">{{pillar.weight}}</div>
          <div style="font-size:10px;color:#475569">{{pillar.desc}}</div>
        </div>
        {% endfor %}
      </div>
    </div>
  </div>
</div>

<!-- ═══════ MODELS TAB ═══════ -->
<div id="tab-models" class="section">
  <div class="section-title">PILLAR 3 — BIOMETRIC REFUTATION MODELS</div>
  <div class="card" style="margin-bottom:16px">
    <div class="card-header">MODEL PERFORMANCE + FEATURE IMPORTANCE</div>
    <div class="card-body" style="padding:8px">
      <img class="chart-img" src="data:image/png;base64,{{ chart_models }}">
    </div>
  </div>
  <div class="grid-2">
    <div class="card">
      <div class="card-header">MODEL METRICS SUMMARY</div>
      <table>
        <thead>
          <tr><th>MODEL</th><th>AUC-ROC</th><th>F1 SCORE</th><th>ROLE</th></tr>
        </thead>
        <tbody>
          {% for m in model_rows %}
          <tr>
            <td style="color:var(--bright);font-family:var(--mono)">{{ m.name }}</td>
            <td>
              <span style="color:var(--{{ m.auc_cls }})">
                {{ m.auc }}
              </span>
            </td>
            <td style="color:var(--text)">{{ m.f1 }}</td>
            <td><span class="badge badge-blue">{{ m.role }}</span></td>
          </tr>
          {% endfor %}
          <tr>
            <td style="color:var(--bright);font-family:var(--mono)">Isolation Forest</td>
            <td><span style="color:var(--purple)">Anomaly</span></td>
            <td style="color:var(--text)">Unsupervised</td>
            <td><span class="badge badge-amber">ANOMALY DET.</span></td>
          </tr>
        </tbody>
      </table>
    </div>
    <div class="card">
      <div class="card-header">DECISION TREE LOGIC (REASON CODES)</div>
      <div class="card-body">
        <div style="font-size:11px;line-height:1.9;color:var(--text)">
          <div style="color:var(--blue)">IF age_delta ≤ 2.95yr</div>
          <div style="padding-left:16px;color:#475569">AND gender_match = 0 → <span style="color:var(--green)">FALSE POSITIVE</span></div>
          <div style="padding-left:16px;color:#475569">AND gender_match = 1</div>
          <div style="padding-left:32px;color:#475569">AND height_delta ≤ 4.45cm → <span style="color:var(--red)">TRUE MATCH</span></div>
          <div style="padding-left:32px;color:#475569">AND height_delta &gt; 4.45cm → context check</div>
          <div style="color:var(--blue);margin-top:8px">ELSE age_delta &gt; 2.95yr</div>
          <div style="padding-left:16px;color:#475569">AND sbert &gt; 0.92 → <span style="color:var(--red)">TRUE MATCH</span></div>
          <div style="padding-left:16px;color:#475569">AND sbert ≤ 0.92 → <span style="color:var(--green)">FALSE POSITIVE</span></div>
          <div style="margin-top:12px;padding:10px;background:#040810;border:1px solid var(--border);border-radius:4px">
            <div style="font-size:9px;color:#334155;letter-spacing:2px;margin-bottom:6px">HARD GATES (override all models)</div>
            <div style="font-size:11px">
              <span style="color:var(--red)">●</span> Gender mismatch → refutation = 0.00<br>
              <span style="color:var(--red)">●</span> Age gap &gt; 10yr → refutation = 0.00
            </div>
          </div>
        </div>
      </div>
    </div>
  </div>
</div>

<!-- ═══════ LOG TAB ═══════ -->
<div id="tab-log" class="section">
  <div class="section-title">SCREENING LOG</div>
  <div class="card">
    <div class="card-header">
      RECENT SCREENINGS
      <button class="nav-pill" onclick="clearLog()" style="font-size:8px">CLEAR</button>
    </div>
    <div class="log-row log-header">
      <div class="log-cell" style="flex:1.5">CLIENT NAME</div>
      <div class="log-cell" style="flex:2">MATCHED FUGITIVE</div>
      <div class="log-cell" style="flex:1">SBERT</div>
      <div class="log-cell" style="flex:1">CRIME</div>
      <div class="log-cell" style="flex:1">REFUTATION</div>
      <div class="log-cell" style="flex:1">FINAL RISK</div>
      <div class="log-cell" style="flex:1.2">VERDICT</div>
    </div>
    <div id="log-body">
      <div style="padding:30px;text-align:center;color:#1e2535;font-size:10px;letter-spacing:3px">
        NO SCREENINGS YET — USE LIVE SCREENING TAB
      </div>
    </div>
  </div>
</div>

<script>
  // ── Clock ──────────────────────────────────────────────────────────────────
  function updateClock() {
    document.getElementById('clock').textContent =
      new Date().toISOString().replace('T',' ').slice(0,19) + ' UTC';
  }
  setInterval(updateClock, 1000); updateClock();

  // ── Tabs ───────────────────────────────────────────────────────────────────
  function showTab(name) {
    document.querySelectorAll('.section').forEach(s => s.classList.remove('active'));
    document.querySelectorAll('.nav-pill').forEach(p => p.classList.remove('active'));
    document.getElementById('tab-' + name).classList.add('active');
    event.target.classList.add('active');
  }

  // ── Demo presets ───────────────────────────────────────────────────────────
  function setDemo(name, age, gender, height, eye, hair, marks) {
    document.getElementById('f-name').value   = name;
    document.getElementById('f-age').value    = age;
    document.getElementById('f-gender').value = gender;
    document.getElementById('f-height').value = height;
    document.getElementById('f-eye').value    = eye;
    document.getElementById('f-hair').value   = hair;
    document.getElementById('f-marks').value  = marks;
  }

  // ── Run Screening ──────────────────────────────────────────────────────────
  function runScreening() {
    const btn = document.getElementById('btn-run');
    btn.disabled = true;
    document.getElementById('loading').style.display = 'block';
    document.getElementById('result-area').style.opacity = '0.4';

    const payload = {
      name:    document.getElementById('f-name').value,
      age:     parseFloat(document.getElementById('f-age').value),
      gender:  document.getElementById('f-gender').value,
      height:  parseFloat(document.getElementById('f-height').value),
      eye:     parseInt(document.getElementById('f-eye').value),
      hair:    parseInt(document.getElementById('f-hair').value),
      marks:   parseInt(document.getElementById('f-marks').value),
    };

    fetch('/api/screen', {
      method: 'POST',
      headers: {'Content-Type':'application/json'},
      body: JSON.stringify(payload),
    })
    .then(r => r.json())
    .then(data => {
      renderResult(data);
      addToLog(data);
      btn.disabled = false;
      document.getElementById('loading').style.display = 'none';
      document.getElementById('result-area').style.opacity = '1';
    })
    .catch(err => {
      console.error(err);
      btn.disabled = false;
      document.getElementById('loading').style.display = 'none';
    });
  }

  // ── Render result ──────────────────────────────────────────────────────────
  function renderResult(d) {
    const vcls = d.verdict.replace(' ','-');
    const vcolor = {
      'HIGH-RISK':'#ff2d2d','REVIEW':'#f59e0b',
      'REFUTED':'#00ff88','AUTO-CLEARED':'#38bdf8'
    }[vcls] || '#94a3b8';

    const riskPct = (d.final_risk * 100).toFixed(1);

    const top3html = d.top3.map((t,i) => `
      <li class="top3-item">
        <span class="top3-rank">#${i+1}</span>
        <span class="top3-name">${t.name}</span>
        <span class="top3-crime">${t.crime} · ${t.country}</span>
        <span class="top3-score" style="color:${t.score>=0.75?'#ff2d2d':t.score>=0.50?'#f59e0b':'#64748b'}">
          ${t.score.toFixed(4)}
        </span>
      </li>`).join('');

    const badgeClass = {
      'HIGH-RISK':'badge-red','REVIEW':'badge-amber',
      'REFUTED':'badge-green','AUTO-CLEARED':'badge-blue'
    }[vcls] || 'badge-blue';

    document.getElementById('result-area').innerHTML = `
      <div class="result-verdict ${vcls}">
        <div>
          <span class="verdict-tag">${d.verdict}</span>
          <div class="result-name">${d.matched_name}</div>
          <div class="result-sub">${d.matched_crime.toUpperCase()} · ${d.matched_country?.toUpperCase() || ''} · ${d.matched_cluster}</div>
        </div>
        <div style="display:flex;align-items:baseline;gap:8px;margin-top:12px">
          <div class="result-score" style="color:${vcolor}">${riskPct}%</div>
          <div style="font-size:11px;color:#475569">FINAL RISK SCORE</div>
        </div>
        <div class="result-reason">${d.reason}</div>
      </div>

      <div class="metric-row">
        <div class="metric-box">
          <div class="metric-label">SBERT SCORE</div>
          <div class="metric-val" style="color:#38bdf8">${d.sbert_score.toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">CRIME WEIGHT</div>
          <div class="metric-val" style="color:#ff2d2d">${d.crime_weight.toFixed(2)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">REFUTATION</div>
          <div class="metric-val" style="color:#f59e0b">${d.refutation.toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">AGE Δ / HT Δ</div>
          <div class="metric-val" style="color:#94a3b8;font-size:13px">${d.age_delta}yr / ${d.height_delta}cm</div>
        </div>
      </div>

      <div class="card" style="margin-bottom:12px">
        <div class="card-header">MODEL CONSENSUS</div>
        <div style="padding:12px 16px;display:grid;grid-template-columns:repeat(4,1fr);gap:8px">
          ${['GB≈XGB','RF','LR','SVM'].map((m,i) => {
            const val = [d.gb,d.rf,d.lr,d.svm][i];
            const c = val>=0.70?'#ff2d2d':val>=0.45?'#f59e0b':'#34d399';
            return `<div style="text-align:center">
              <div style="font-size:9px;color:#334155;margin-bottom:6px">${m}</div>
              <div style="font-size:18px;color:${c};font-family:var(--mono)">${(val*100).toFixed(1)}%</div>
            </div>`;
          }).join('')}
        </div>
      </div>

      <div class="card">
        <div class="card-header">TOP 3 MATCHES — PILLAR 1 SBERT</div>
        <div style="padding:8px 16px">
          <ul class="top3-list">${top3html}</ul>
        </div>
      </div>
    `;

    // Show result chart
    if (d.chart_b64) {
      document.getElementById('screening-chart').style.display = 'block';
      document.getElementById('result-chart-img').src = 'data:image/png;base64,' + d.chart_b64;
    }
  }

  // ── Screening log ──────────────────────────────────────────────────────────
  let logEntries = [];

  function addToLog(d) {
    logEntries.unshift(d);
    renderLog();
  }

  function renderLog() {
    if (!logEntries.length) return;
    const vcls = {
      'HIGH RISK':'badge-red','REVIEW':'badge-amber',
      'REFUTED':'badge-green','AUTO-CLEARED':'badge-blue'
    };
    const html = logEntries.map(d => `
      <div class="log-row">
        <div class="log-cell" style="flex:1.5;color:var(--bright)">${d.client_name}</div>
        <div class="log-cell" style="flex:2;color:#64748b">${d.matched_name}</div>
        <div class="log-cell" style="flex:1;color:#38bdf8">${d.sbert_score.toFixed(4)}</div>
        <div class="log-cell" style="flex:1;color:#64748b;font-size:10px">${d.matched_crime}</div>
        <div class="log-cell" style="flex:1;color:#f59e0b">${d.refutation.toFixed(4)}</div>
        <div class="log-cell" style="flex:1;color:${d.final_risk>=0.70?'#ff2d2d':d.final_risk>=0.45?'#f59e0b':'#34d399'}">
          ${(d.final_risk*100).toFixed(1)}%
        </div>
        <div class="log-cell" style="flex:1.2">
          <span class="badge ${vcls[d.verdict]||'badge-blue'}">${d.verdict}</span>
        </div>
      </div>`).join('');
    document.getElementById('log-body').innerHTML = html;
  }

  function clearLog() {
    logEntries = [];
    document.getElementById('log-body').innerHTML =
      '<div style="padding:30px;text-align:center;color:#1e2535;font-size:10px;letter-spacing:3px">LOG CLEARED</div>';
  }

  // ── Enter key triggers screening ───────────────────────────────────────────
  document.addEventListener('keydown', e => {
    if (e.key === 'Enter' &&
        document.getElementById('tab-screening').classList.contains('active')) {
      runScreening();
    }
  });
</script>
</body>
</html>
"""

# ─────────────────────────────────────────────────────────────────────────────
# FLASK APP
# ─────────────────────────────────────────────────────────────────────────────

app = Flask(__name__)

MODEL_ROWS = [dict(
    name    = n,
    auc     = f"{p3[n]['auc']:.4f}",
    auc_raw = float(p3[n]['auc']),
    f1      = f"{p3[n]['f1']:.4f}",
    role    = ('Primary'       if n=='GB' else
               'Ensemble'      if n=='RF' else
               'Probabilistic' if n=='LR' else
               'Reason Codes'  if n=='DT' else 'High-Dim'),
    auc_cls = ('green' if p3[n]['auc'] >= 0.999 else
               'blue'  if p3[n]['auc'] >= 0.99  else 'amber'),
) for n in p3]

@app.route('/')
def index():
    return render_template_string(HTML,
        kpi          = KPI,
        chart_crime  = CHART_CRIME,
        chart_cluster= CHART_CLUSTER,
        chart_models = CHART_MODELS,
        chart_age    = CHART_AGE,
        cluster_table= CLUSTER_TABLE,
        model_rows   = MODEL_ROWS,
    )

@app.route('/api/screen', methods=['POST'])
def api_screen():
    data = request.get_json()
    try:
        result = screen_client(
            client_name    = str(data.get('name','Unknown')),
            client_age     = float(data.get('age', 35)),
            client_gender  = str(data.get('gender','M')),
            client_height  = float(data.get('height', 170)),
            client_eye_enc = int(data.get('eye', 1)),
            client_hair_enc= int(data.get('hair', 1)),
            marks_match    = int(data.get('marks', 0)),
        )
        # Generate and attach screening chart
        result['chart_b64'] = chart_screening_result(result)
        # Add log to global list
        screening_log.append(result)
        return jsonify(result)
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/stats')
def api_stats():
    return jsonify(dict(
        total_fugitives = len(db),
        screening_count = len(screening_log),
        km_silhouette   = round(km_sil, 4),
        gb_auc          = round(p3['GB']['auc'], 4),
    ))

if __name__ == '__main__':
    app.run(debug=False, host='127.0.0.1', port=5050)


  INTERPOL 2026 DASHBOARD — STARTUP
✅  Loaded 7,057 fugitives from /Users/dreyw/SMU_School/Term4/Project/Interpol/outputs/interpol_final_fugitive_db.csv
⚙  Building Pillar 1 name embeddings...
   Name embeddings: (7057, 5000)
⚙  Building Pillar 2 crime cluster embeddings...
   K-Means silhouette: 0.370
⚙  Training Pillar 3 biometric refutation models...
   Models trained: ['LR', 'DT', 'RF', 'GB', 'SVM']
   GB AUC: 1.0000 | RF AUC: 1.0000

✅  All pillars ready. Starting Flask server...
   Open browser at: http://127.0.0.1:5050

⚙  Pre-rendering charts...
✅  Charts ready.

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5050
Press CTRL+C to quit
